<a href="https://colab.research.google.com/github/Muhammadjon1211/jaydariGPT/blob/master/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch bitsandbytes datasets accelerate peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 57.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]


In [2]:
%env TRL_USE_FUSED_LORA=False

import os
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments, pipeline
import torch
from datasets import load_dataset
from peft import LoraModel, LoraConfig, get_peft_model
from trl import SFTTrainer

env: TRL_USE_FUSED_LORA=False


In [24]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    # dtype=torch.float16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [25]:
# Before fine-tuning

prompt = "Explain what a tokenizer is."
#prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7,
  )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

#print(model)
first_block = model.model.layers[0]
print("first_block:", first_block)
print("=============")
print(first_block.self_attn)
print("=============")
print(model.config)
print("=============")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is.
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_feat

In [26]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parametes (including frozen 4-bit): {total_params:,}")

Total parametes (including frozen 4-bit): 615,606,272


In [27]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
# dataset
dataset[1]

{'output': 'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).',
 'input': '',
 'instruction': 'What are the three primary colors?'}

In [28]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_text}"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}"
    )

#generate_prompt(dataset[0])

def formatting_func(example):
  return {"text": generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [29]:
dataset = dataset.select(range(5000))

In [30]:
dataset = dataset.shuffle(seed=42)

In [31]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine-tuning
# OOM => Out of memory

# ADAPTERS
# => LoRa (Low Rank Adaptation)

In [32]:
lora_config = LoraConfig(
    r=8, #rank
    lora_alpha=16,
    lora_dropout=0.05, # reduce overfitting
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

In [33]:
# Wrap the model exactly once
model = get_peft_model(model, lora_config)

In [34]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [35]:
from trl import SFTConfig

training_args = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # VRAM optimization
    num_train_epochs=2,
    logging_steps=20,
    output_dir="./jaydari_gpt",
    save_strategy="epoch",
    bf16=True,
    fp16=False, # T4 mixed precision
    report_to="none",
)

In [36]:
import trl.kernels.chunked_logprob
# Dynamically patch TRL's loaded module to disable the incompatible fused matrix multiplication on T4 GPU
trl.kernels.chunked_logprob._MM_OUT_DTYPE = False

# SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset = dataset,
    formatting_func=lambda x: x['text'],
    args=training_args,
)

trainer.train()
model.save_pretrained('jaydari_gpt')
tokenizer.save_pretrained('jaydari_gpt')

Applying formatting function to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Step,Training Loss
20,1.512762
40,1.546455
60,1.517977
80,1.531450
100,1.461786
120,1.415787
140,1.448229
160,1.394619
180,1.406501
200,1.322576


('jaydari_gpt/tokenizer_config.json',
 'jaydari_gpt/chat_template.jinja',
 'jaydari_gpt/tokenizer.json')

In [37]:
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(base_model_id, device_map='auto')

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [38]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
  output_base = model_base.generate(
      **inputs_base,
      max_new_tokens=120,
      do_sample=True,
      temperature=0.7,
  )

print(tokenizer_base.decode(output_ids[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is.


In [40]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 16.5 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [43]:
model_path = 'jaydari_gpt'

tokenizer_ft = AutoTokenizer.from_pretrained(model_path)
model_ft = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map='auto'
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [44]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

inputs_ft = tokenizer_ft(prompt, return_tensors='pt').to(model_ft.device)

with torch.no_grad():
  output_ft = model_ft.generate(
      **inputs_ft,
      max_new_tokens=120,
      do_sample=True,
      temperature=0.7,
  )

print("===== FINE-TUNED MODEL OUTPUT =====")
print(tokenizer_ft.decode(output_ft[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FINE-TUNED MODEL OUTPUT =====
### Instruction:
Explain what machine learning is in simple words.

### Response:
Machine learning is the field of artificial intelligence that aims to develop algorithms that can learn from data and improve their predictions based on past experiences. It is a powerful tool for optimizing tasks and building intelligent systems that can interact with data and make decisions. In simple terms, machine learning is the process of training a computer to learn from data. The goal is to allow the machine to make predictions based on the data it has encountered.


In [45]:
!zip -r jaydari_gpt.zip jaydari_gpt

  adding: jaydari_gpt/ (stored 0%)
  adding: jaydari_gpt/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/checkpoint-1250/ (stored 0%)
  adding: jaydari_gpt/checkpoint-1250/scheduler.pt (deflated 62%)
  adding: jaydari_gpt/checkpoint-1250/rng_state.pth (deflated 26%)
  adding: jaydari_gpt/checkpoint-1250/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/checkpoint-1250/tokenizer_config.json (deflated 46%)
  adding: jaydari_gpt/checkpoint-1250/training_args.bin (deflated 53%)
  adding: jaydari_gpt/checkpoint-1250/README.md (deflated 65%)
  adding: jaydari_gpt/checkpoint-1250/optimizer.pt (deflated 22%)
  adding: jaydari_gpt/checkpoint-1250/chat_template.jinja (deflated 60%)
  adding: jaydari_gpt/checkpoint-1250/adapter_model.safetensors (deflated 24%)
  adding: jaydari_gpt/checkpoint-1250/trainer_state.json (deflated 79%)
  adding: jaydari_gpt/checkpoint-1250/adapter_config.json (deflated 60%)
  adding: jaydari_gpt/tokenizer_config.json (deflated 46%)
  adding: jaydari_gpt/README.m

In [46]:
from google.colab import files
files.download("jaydari_gpt.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
###############################################
#              MODEL TESTING                  #
###############################################